## **Farmland Birds Under Pressure**
# Biodiversity Trends and Agricultural Change in Europe

**Main question**

**How have common farmland-bird populations changed across European countries since 2000, and are agricultural intensity, organic farming and protected-area coverage associated with different national trajectories?**

**Layer 1:** Long-term biodiversity trends

Use:

- env_bio2: national farmland-bird indices
- env_bio3: EU bird-group indices and uncertainty

Source: https://ec.europa.eu/eurostat/web/main/data/database#Updates


**Period:**

1990–2019 for Germany and long-term countries
2000–2019 for the main cross-country comparison
Later years only in a clearly labelled supplementary analysis

**Questions:**

- Which countries experienced the largest decline since 2000?
- Did Germany decline faster than the EU benchmark?
- Are farmland birds declining faster than forest birds?
- Are there countries showing stabilization or recovery?
- How sensitive are rankings to the chosen endpoint?

 **Bird groups**
CO_ALL: all common bird species
CO_FARM: common farmland bird species
CO_FOR: common forest bird species

**Estimate types**
NSME: unsmoothed estimate
SME: smoothed estimate
SME_LW95: lower 95% confidence limit
SME_UP95: upper 95% confidence limit

 **Index reference systems**
I00: 2000 = 100
I90: 1990 = 100
I_LY: latest year = 100


**Layer 2: Environmental drivers**
Add three Eurostat indicators.

1. Organic farming

Dataset:[Area under organic farming — sdg_02_40](https://ec.europa.eu/eurostat/databrowser/view/tag00025/default/table?lang=en)

In [5]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
df_farmland_birds = pd.read_csv('/content/drive/MyDrive/Final_Project/estat_env_bio2.tsv.gz', sep="\t", compression="gzip")
df_farmland_birds.head(20)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,"freq,unit,geo\TIME_PERIOD",1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,"A,I00,AT",:,:,:,:,:,:,:,:,100.00,...,62.90 d,58.40 d,60.70 d,55.20 d,61.50 d,61.70 d,60.50 d,:,:,:
1,"A,I00,BE",149.09,142.38,136.17,130.38,124.98,119.95,115.32,111.08,107.13,...,67.96,65.53,63.18,60.93,58.75,56.62,:,:,:,:
2,"A,I00,CH",118.92,112.30,113.12,114.63,114.98,112.34,108.66,100.62,96.47,...,100.00,101.87,101.00,99.48,105.65,117.65,119.19,115.61,116.39,:
3,"A,I00,CY",:,:,:,:,:,:,:,:,:,...,117.00 d,132.00 d,110.00 d,117.00 d,123.00 d,120.00 d,110.00 d,81.00 d,:,:
4,"A,I00,CZ",112.68,121.44,129.46,135.42,138.75,138.06,130.19,117.86,106.83,...,74.37,72.54,70.72,68.93,67.16,65.41,63.67,61.94,60.23,:
5,"A,I00,DE",143.37,136.48,126.61,134.59,135.08,116.73,121.72,107.02,100.38,...,60.84 d,61.09 d,54.84 d,53.46 d,54.72 d,:,:,:,:,:
6,"A,I00,DK",129.45,118.91,110.40,109.81,111.84,109.54,109.09,103.51,101.78,...,80.05,78.49,74.13,69.77,67.27,76.10,72.38,61.85,66.84,:
7,"A,I00,EE",122.15,111.09,113.72,104.89,94.64,82.13,93.46,98.66,94.43,...,84.42 d,77.11 d,76.10 d,74.07 d,74.71 d,68.99 d,:,:,:,:
8,"A,I00,EL",:,:,:,:,:,:,:,:,:,...,90.91 d,90.79 d,93.51 d,89.34 d,95.11 d,88.61 d,117.18 d,86.76 d,83.31 d,:
9,"A,I00,ES",:,:,:,:,:,:,:,:,107.39,...,77.53,78.04,74.55,69.73,72.39,71.15,70.77,69.23,72.64,:


In [8]:
#1. # Checking for Null Values
df_farmland_birds.isnull().sum()

,0
"freq,unit,geo\TIME_PERIOD",0
1990,0
1991,0
1992,0
1993,0
1994,0
1995,0
1996,0
1997,0
1998,0


In [12]:
#NO duplicates
df_farmland_birds.duplicated().sum()

np.int64(0)

In [11]:
import pandas as pd

id_column = df_farmland_birds.columns[0]

long = df_farmland_birds.melt(
    id_vars=id_column,
    var_name="year",
    value_name="raw_value"
)

long[["frequency", "unit", "country"]] = (
    long[id_column]
    .str.split(",", expand=True)
)

long["year"] = long["year"].str.strip().astype(int)

long["value"] = pd.to_numeric(
    long["raw_value"].str.extract(r"([-+]?\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

long["flag"] = (
    long["raw_value"]
    .str.extract(r"[-+]?\d+(?:\.\d+)?\s*(.*)")[0]
    .str.strip()
    .replace("", pd.NA)
)

long = long[
    ["country", "year", "value", "flag", "frequency", "unit"]
]

long.head()

,country,year,value,flag,frequency,unit
0,AT,1990,NaN,NaN,A,I00
1,BE,1990,149.09,<NA>,A,I00
2,CH,1990,118.92,<NA>,A,I00
3,CY,1990,NaN,NaN,A,I00
4,CZ,1990,112.68,<NA>,A,I00


env_bio3: very useful

This is the most valuable new file. Eurostat calls it Common bird indices by type of estimate.

It contains 36 time series formed from:

Bird groups
CO_ALL: all common bird species
CO_FARM: common farmland bird species
CO_FOR: common forest bird species
Estimate types
NSME: unsmoothed estimate
SME: smoothed estimate
SME_LW95: lower 95% confidence limit
SME_UP95: upper 95% confidence limit
Index reference systems
I00: 2000 = 100
I90: 1990 = 100
I_LY: latest year = 100

The file has no missing values: 1,260 observations out of 1,260 possible values.
I will select:
- geo      = EU27_2020
- unit     = I00
-statinfo = SME
-comspec  = CO_FARM


In [2]:

df_birds_species = pd.read_csv('/content/drive/MyDrive/Final_Project/estat_env_bio3.tsv.gz', sep="\t", compression="gzip")

df_birds_species.head(20)

,"freq,statinfo,comspec,unit,geo\TIME_PERIOD",1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,"A,NSME,CO_ALL,I00,EU27_2020",107.81,106.83,105.44,106.55,107.54,103.76,102.08,101.06,97.54,...,92.38,91.54,90.84,83.44,86.79,87.86,86.76,84.27,85.58,85.38
1,"A,NSME,CO_ALL,I90,EU27_2020",105.26,104.29,102.94,103.96,104.91,101.19,99.59,98.60,95.17,...,90.23,89.33,88.75,81.39,84.72,85.82,84.62,82.24,83.59,83.24
2,"A,NSME,CO_ALL,I_LY,EU27_2020",127.11,125.87,124.21,125.54,126.80,122.12,120.23,119.12,114.83,...,108.93,107.91,107.13,98.30,102.31,103.52,102.23,99.27,100.89,100.51
3,"A,NSME,CO_FARM,I00,EU27_2020",113.90,112.14,117.47,116.36,113.24,105.71,106.21,105.40,101.79,...,81.61,82.51,75.09,70.92,73.30,74.30,73.17,67.73,67.43,69.94
4,"A,NSME,CO_FARM,I90,EU27_2020",98.54,97.17,101.67,100.81,98.26,91.53,91.78,91.57,88.18,...,70.17,71.42,64.81,61.17,63.50,64.18,63.24,58.62,58.19,60.45
5,"A,NSME,CO_FARM,I_LY,EU27_2020",168.24,165.99,173.09,171.42,167.05,155.83,156.25,155.73,150.33,...,120.14,122.11,110.87,104.69,108.43,109.57,107.83,99.80,99.19,103.16
6,"A,NSME,CO_FOR,I00,EU27_2020",120.43,118.99,112.33,103.98,110.87,102.38,103.68,111.24,99.05,...,96.88,94.88,100.19,88.81,93.67,96.16,95.90,95.52,101.93,94.71
7,"A,NSME,CO_FOR,I90,EU27_2020",113.39,111.80,105.68,97.82,104.23,96.25,97.47,104.40,93.37,...,90.89,89.34,94.16,83.50,88.09,90.28,90.15,89.83,95.84,89.01
8,"A,NSME,CO_FOR,I_LY,EU27_2020",122.71,121.30,114.25,105.72,112.73,104.04,105.37,113.02,100.90,...,98.50,96.63,101.73,90.31,95.34,97.74,97.46,97.10,103.63,96.37
9,"A,SME,CO_ALL,I00,EU27_2020",102.47,102.69,102.83,102.80,102.62,102.38,102.05,101.66,101.21,...,89.44,88.90,88.37,87.84,87.34,86.82,86.33,85.84,85.36,84.88


In [9]:
#1. # Checking for Null Values
df_birds_species.isnull().sum()

,0
"freq,statinfo,comspec,unit,geo\TIME_PERIOD",0
1990,0
1991,0
1992,0
1993,0
1994,0
1995,0
1996,0
1997,0
1998,0


env_bio4: useful, with limitations

Eurostat identifies this as Protected areas.

It distinguishes:

TPA: terrestrial protected area
MPA: marine protected area
KM2: square kilometres
PC: percentage

For the bird project, use only:

areaprot = TPA
unit     = PC

Marine protected areas are not a plausible direct explanatory variable for farmland birds.

For Germany, terrestrial protected-area coverage rises from:

37.1% in 2011
37.8% in 2019
39.1% in 2023
Important limitation

Protected-area percentage changes slowly. It may explain differences between countries, but probably not annual fluctuations in bird populations.

In [4]:
drive.mount('/content/drive')
df_protected_areas = pd.read_csv('/content/drive/MyDrive/Final_Project/estat_env_bio4.tsv.gz', sep="\t", compression="gzip")
df_protected_areas.head(20)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,"freq,unit,areaprot,geo\TIME_PERIOD",2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,"A,KM2,MPA,AT",:,: m,:,:,:,: m,:,:,: m,:,: m,: m,: m
1,"A,KM2,MPA,BE",:,1273,:,:,:,1274,:,:,1274,:,1322,1320 b,1332
2,"A,KM2,MPA,BG",:,995,:,:,:,2834,:,:,2834,:,2856,2857 b,2891
3,"A,KM2,MPA,CY",:,131,:,:,:,131,:,:,8463,:,8463,8481 b,8518
4,"A,KM2,MPA,CZ",:,: m,:,:,:,: m,:,:,: m,:,: m,: m,: m
5,"A,KM2,MPA,DE",:,25641,:,:,:,25684,:,:,25688,:,25637,25616 b,25739
6,"A,KM2,MPA,DK",:,19087,:,:,:,19122,:,:,19675,:,19681,19664 b,25005
7,"A,KM2,MPA,EE",:,6759,:,:,:,6757,:,:,6813,:,6813,6814 b,7108
8,"A,KM2,MPA,EL",:,7414,:,:,:,7430,:,:,22550,:,22746,22771 b,23163
9,"A,KM2,MPA,ES",:,11489,:,:,:,84387,:,:,128476,:,132818,132934 b,196632


In [10]:
#1. # Checking for Null Values
df_protected_areas.isnull().sum()

,0
"freq,unit,areaprot,geo\TIME_PERIOD",0
2011,0
2012,0
2013,0
2014,0
2015,0
2016,0
2017,0
2018,0
2019,0
